# Vessel Segmentation
Recommended datasets: **DRIVE**, **STARE**, and **CHASE-DB1** (combine for a medium-sized vessel corpus).
This notebook demonstrates preprocessing for retinal vessel segmentation tasks.


In [ ]:
import sys
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
import sklearn

print("python", sys.version)
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("opencv-python", cv2.__version__)
print("matplotlib", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)


## Repository setup
This notebook expects the repository `src` folder on the Python path.


In [ ]:
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from engine.image_preprocessing import PreprocessConfig, preprocess_fundus_image

data_dir = repo_root / "data"
raw_dir = data_dir / "raw"
processed_dir = data_dir / "processed"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)


## KaggleHub setup (Colab)
Upload your `kaggle.json` file or set `KAGGLE_USERNAME` and `KAGGLE_KEY` before using KaggleHub.


In [ ]:
import sys
import subprocess
from pathlib import Path

try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "kagglehub"], check=True)
    import kagglehub

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if not kaggle_json.exists():
    print("Upload kaggle.json to ~/.kaggle or set KAGGLE_USERNAME/KAGGLE_KEY.")
    print("Colab tip: use files.upload() and then move the file.")

if kaggle_json.exists():
    kaggle_json.chmod(0o600)


## Download vessel datasets
KaggleHub downloads each dataset into `data/raw/vessels/{drive,stare,chase_db1}`.


In [ ]:
import shutil
from pathlib import Path

vessel_slugs = {
    "drive": "andrewmvd/drive-dataset",
    "stare": "andrewmvd/stare-dataset",
    "chase_db1": "andrewmvd/chasedb1-dataset",
}
vessel_root = raw_dir / "vessels"
vessel_root.mkdir(parents=True, exist_ok=True)

if not any(vessel_root.rglob("*.png")) and not any(vessel_root.rglob("*.jpg")):
    for name, slug in vessel_slugs.items():
        dataset_path = Path(kagglehub.dataset_download(slug))
        target_path = vessel_root / name
        if dataset_path.exists() and dataset_path != target_path:
            shutil.copytree(dataset_path, target_path, dirs_exist_ok=True)
    print("Vessel data ready at", vessel_root)
else:
    print("Using existing vessel data at", vessel_root)


## Index vessel images and masks
The helper below pairs each image with the closest mask sharing a stem.


In [4]:
from sklearn.model_selection import train_test_split

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".gif"}

def _is_mask_path(path):
    tokens = " ".join([path.name.lower()] + [p.name.lower() for p in path.parents])
    return any(key in tokens for key in ["mask", "manual", "groundtruth", "gt", "seg"])

def find_images_and_masks(root):
    image_files = [p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS and not _is_mask_path(p)]
    mask_files = [p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS and _is_mask_path(p)]
    mask_map = {mask.stem: mask for mask in mask_files}
    rows = []
    for image in sorted(image_files):
        mask = mask_map.get(image.stem)
        rows.append({"image_path": image, "mask_path": mask})
    return pd.DataFrame(rows)

vessel_df = find_images_and_masks(vessel_root)
print("Images:", len(vessel_df), "with masks:", vessel_df["mask_path"].notnull().sum())

if not vessel_df.empty:
    train_df, temp_df = train_test_split(vessel_df, test_size=0.3, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))


## Preprocess and visualize a sample


In [ ]:
if not vessel_df.empty:
    sample = vessel_df.iloc[0]
    image_array = cv2.imread(str(sample["image_path"]))
    if image_array is None:
        print("Failed to read", sample["image_path"])
    else:
        config = PreprocessConfig(target_size=(512, 512), normalization="zero_one")
        result = preprocess_fundus_image(sample["image_path"], mask=sample["mask_path"], config=config)
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        original = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
        axes[0].imshow(original)
        axes[0].set_title("Original")
        axes[0].axis("off")
        axes[1].imshow(result.image)
        axes[1].set_title("Preprocessed")
        axes[1].axis("off")
        if result.mask is not None:
            axes[2].imshow(result.mask, cmap="gray")
            axes[2].set_title("Vessel Mask")
            axes[2].axis("off")
        else:
            axes[2].axis("off")
        plt.tight_layout()


## Next step
Train a vessel segmentation model using the split DataFrames and log vessel probability maps.
